# Cross-Domain LOS/NLOS Classification Using Deep CORAL with Semi-Supervised Fine-Tuning

## Introduction and Problem Statement

Reliable **line-of-sight (LOS)** versus **non-line-of-sight (NLOS)** identification is essential for accurate indoor localization and wireless communication systems. Machine learning models trained on channel impulse response (CIR) and related tabular features can achieve high accuracy in a single environment. However, when deployed in a new environment (domain) with different hardware, propagation conditions, or measurement protocols, performance often degrades severely.  
This project addresses **domain generalization** for LOS/NLOS classification: the model is trained on two source domains and must perform well on a completely unseen target domain.

## Data Description and Domain Shift

The dataset comprises measurements from three independent research campaigns, each representing a distinct domain.

| Domain | Total Samples | LOS % | NLOS % |
|--------|---------------|-------|--------|
| 1      | 42,000        | 50.00 | 50.00  |
| 2      | 20,360        | 32.12 | 67.88  |
| 3      | 4,796         | 63.99 | 36.01  |

For each sample, a **300‑point CIR vector** and **14 tabular features** (e.g., range, received power, noise estimates) are provided.

A critical challenge emerged during exploratory analysis: **12 of the 14 tabular features in Domain 3 are constant `-1`**, indicating missing or placeholder values. In the source domains, these features carry useful signal, but they provide no information in the target domain. Additionally, the class distributions differ significantly across domains. This severe domain shift and feature missingness explain why conventional training methods failed to generalize.

## Approach Rationale

Initial experiments with **adversarial domain adaptation (DANN)** and **hybrid CORAL+DANN** were unstable and did not improve target accuracy. We therefore focused on **Deep CORAL (correlation alignment)**, which aligns the second‑order statistics (covariances) of source and target feature distributions. This method is simpler, more stable, and directly addresses the distribution mismatch without requiring a domain classifier.

Because a small number of labeled target samples (50–200) were available, we also employed **semi‑supervised fine‑tuning**: after the unsupervised CORAL training, the model was fine‑tuned on a combination of all source data and the few labeled target samples. This adapts the decision boundary to the target domain while leveraging the features learned from abundant source data.

## Preprocessing and Model Architecture

- **CIR normalization:** Each CIR sample is normalized by its maximum absolute amplitude to remove device‑specific scaling.
- **Tabular feature standardization:** Standardization parameters (mean, standard deviation) are computed only on the source domains to prevent target data leakage.
- **Placeholder handling:** Features that are constant `-1` in Domain 3 are kept as‑is (no imputation or removal), as experiments showed that preserving the original values yielded the best target accuracy.

The model consists of two input branches:

1. **CIR branch:** A 1D convolutional neural network with residual blocks (filters 32→64→128) and global average pooling extracts a 64‑dimensional feature vector from the 300×1 CIR input.
2. **Tabular branch:** A fully connected network (64→64) processes the 14 tabular features.

The two feature vectors are concatenated and passed through additional dense layers (128→64) to produce a shared representation. A task classifier (64→32→1) predicts the LOS/NLOS probability. During training, **CORAL loss** is computed between the source and target feature covariances and added to the binary cross‑entropy task loss with a time‑dependent weight.

## Training Procedure

- **Unsupervised domain adaptation:** The model is trained on labeled source data and unlabeled target data using CORAL loss to align features. A cosine learning rate schedule and early stopping based on source validation accuracy are used.
- **Semi‑supervised fine‑tuning:** After CORAL training, the entire model is fine‑tuned for 10 epochs on source plus a small labeled subset from the target domain. A low learning rate (`1e-5`) is used to avoid catastrophic forgetting.
- **Ensemble and test‑time augmentation (TTA):** To improve robustness, three models with different random seeds are trained. At inference, each model’s predictions are averaged over five augmented versions of the test input (small Gaussian noise and random time shifts), and the ensemble averages the outputs of all models.

## Results and Interpretation

The final ensemble achieved **approximately 74–76% accuracy** on the unseen target domain, while the source validation accuracy was around **93%**. The large gap between source and target performance reflects the severe domain shift and the presence of placeholder features in the target domain. Despite this, the model significantly outperforms chance (50%) and provides a usable baseline for cross‑domain LOS/NLOS classification.

## Impact and Future Work

This project demonstrates that Deep CORAL combined with limited target labeling can partially mitigate domain shift, but the achievable accuracy is bounded by the intrinsic differences between datasets. The results highlight the importance of consistent feature collection across measurement campaigns.

Future work could explore:

- Collecting more balanced and complete target‑domain data.
- Explicitly modeling missing features (e.g., missingness indicators) to handle placeholders.
- Applying advanced domain adaptation techniques (e.g., maximum classifier discrepancy, self‑training with pseudo‑labels).
- Using transfer learning from pre‑trained wireless signal models.

The code is available as a self‑contained pipeline and can be adapted to other cross‑domain classification tasks facing similar data heterogeneity.

In [ ]:
# ============================================================
# ENSEMBLE OF DEEP CORAL + ORIGINAL SIMPLE FINE-TUNING
# NUM_LABELED = 200, WITH TTA
# ============================================================
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision
from sklearn.preprocessing import StandardScaler

mixed_precision.set_global_policy('mixed_float16')

# ------------------ CORAL Model (original) ------------------
class CORALUDAModel(keras.Model):
    def __init__(self, feature_extractor, classifier, max_coral_weight=500.0, max_entropy_weight=0.0):
        super().__init__()
        self.feature_extractor = feature_extractor
        self.classifier = classifier
        self.max_coral_weight = max_coral_weight
        self.max_entropy_weight = max_entropy_weight
        self.current_epoch = tf.Variable(0.0, trainable=False, dtype=tf.float32)
        self.total_epochs = tf.Variable(50.0, trainable=False, dtype=tf.float32)
        self.task_loss_tracker = keras.metrics.Mean(name="task_loss")
        self.coral_loss_tracker = keras.metrics.Mean(name="coral_loss")
        self.entropy_loss_tracker = keras.metrics.Mean(name="entropy_loss")
        self.acc_tracker = keras.metrics.BinaryAccuracy(name="accuracy")

    @property
    def metrics(self):
        return [self.task_loss_tracker, self.coral_loss_tracker, self.entropy_loss_tracker, self.acc_tracker]

    def set_epoch(self, epoch, total_epochs):
        self.current_epoch.assign(float(epoch))
        self.total_epochs.assign(float(total_epochs))

    def compute_coral_loss(self, source_feats, target_feats):
        src = tf.cast(source_feats, tf.float32)
        tgt = tf.cast(target_feats, tf.float32)
        d = tf.cast(tf.shape(src)[1], tf.float32)
        src_centered = src - tf.reduce_mean(src, axis=0, keepdims=True)
        tgt_centered = tgt - tf.reduce_mean(tgt, axis=0, keepdims=True)
        n_src = tf.cast(tf.shape(src)[0], tf.float32)
        n_tgt = tf.cast(tf.shape(tgt)[0], tf.float32)
        cov_src = tf.matmul(src_centered, src_centered, transpose_a=True) / (n_src - 1.0 + 1e-5)
        cov_tgt = tf.matmul(tgt_centered, tgt_centered, transpose_a=True) / (n_tgt - 1.0 + 1e-5)
        return tf.reduce_sum(tf.square(cov_src - cov_tgt)) / (4.0 * d * d)

    def train_step(self, data):
        (src_cir, src_tab, y_src), (tgt_cir, tgt_tab) = data
        p = self.current_epoch / self.total_epochs
        weight_factor = 2.0 / (1.0 + tf.exp(-10.0 * p)) - 1.0
        coral_weight = self.max_coral_weight * weight_factor
        entropy_weight = self.max_entropy_weight * weight_factor

        with tf.GradientTape() as tape:
            src_feats = self.feature_extractor([src_cir, src_tab], training=True)
            tgt_feats = self.feature_extractor([tgt_cir, tgt_tab], training=True)
            src_preds = self.classifier(src_feats, training=True)
            tgt_preds = self.classifier(tgt_feats, training=True)

            y_src = tf.cast(y_src, tf.float32)
            y_src = tf.expand_dims(y_src, axis=-1)
            task_loss = tf.reduce_mean(keras.losses.binary_crossentropy(y_src, src_preds))
            coral_loss = self.compute_coral_loss(src_feats, tgt_feats)

            tgt_preds_c = tf.clip_by_value(tf.cast(tgt_preds, tf.float32), 1e-7, 1.0 - 1e-7)
            entropy_loss = -tf.reduce_mean(tgt_preds_c * tf.math.log(tgt_preds_c) + (1 - tgt_preds_c) * tf.math.log(1 - tgt_preds_c))
            total_loss = task_loss + coral_weight * coral_loss + entropy_weight * entropy_loss

        grads = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        self.task_loss_tracker.update_state(task_loss)
        self.coral_loss_tracker.update_state(coral_loss)
        self.entropy_loss_tracker.update_state(entropy_loss)
        self.acc_tracker.update_state(y_src, src_preds)
        return {"task_loss": self.task_loss_tracker.result(),
                "coral_loss": self.coral_loss_tracker.result(),
                "entropy_loss": self.entropy_loss_tracker.result(),
                "accuracy": self.acc_tracker.result()}

    def test_step(self, data):
        (cir, tab), y = data
        feats = self.feature_extractor([cir, tab], training=False)
        preds = self.classifier(feats, training=False)
        y = tf.cast(y, tf.float32)
        y = tf.expand_dims(y, axis=-1)
        task_loss = tf.reduce_mean(keras.losses.binary_crossentropy(y, preds))
        self.acc_tracker.update_state(y, preds)
        return {"task_loss": task_loss, "accuracy": self.acc_tracker.result()}

# ------------------ Backbone ------------------
def resnet_block_1d(x, filters, kernel_size=3, stride=1):
    shortcut = x
    x = layers.Conv1D(filters, kernel_size=kernel_size, strides=stride, padding='same')(x)
    x = layers.LayerNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv1D(filters, kernel_size=kernel_size, strides=1, padding='same')(x)
    x = layers.LayerNormalization()(x)
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv1D(filters, kernel_size=1, strides=stride, padding='same')(shortcut)
        shortcut = layers.LayerNormalization()(shortcut)
    x = layers.add([x, shortcut])
    return layers.Activation('relu')(x)

def build_backbone(tabular_dim):
    reg = keras.regularizers.l2(1e-4)
    cir_input = layers.Input(shape=(300, 1), name='cir_input')
    x_cir = layers.Conv1D(32, kernel_size=7, strides=2, padding='same', kernel_regularizer=reg)(cir_input)
    x_cir = layers.LayerNormalization()(x_cir)
    x_cir = layers.Activation('relu')(x_cir)
    x_cir = resnet_block_1d(x_cir, 32, stride=1)
    x_cir = resnet_block_1d(x_cir, 64, stride=2)
    x_cir = resnet_block_1d(x_cir, 128, stride=2)
    x_cir = layers.GlobalAveragePooling1D()(x_cir)
    cir_feats = layers.Dense(64, activation='relu', kernel_regularizer=reg)(x_cir)

    tab_input = layers.Input(shape=(tabular_dim,), name='tab_input')
    x_tab = layers.Dense(64, activation='relu', kernel_regularizer=reg)(tab_input)
    x_tab = layers.LayerNormalization()(x_tab)
    tab_feats = layers.Dense(64, activation='relu', kernel_regularizer=reg)(x_tab)

    fused = layers.concatenate([cir_feats, tab_feats])
    fused = layers.Dense(128, activation='relu', kernel_regularizer=reg)(fused)
    fused = layers.LayerNormalization()(fused)
    fused = layers.Dropout(0.4)(fused)
    features = layers.Dense(64, activation='relu', kernel_regularizer=reg, name='shared_features')(fused)

    feature_extractor = keras.Model(inputs=[cir_input, tab_input], outputs=features, name='feature_extractor')

    feat_input = layers.Input(shape=(64,), name='feature_input')
    out = layers.Dense(32, activation='relu', kernel_regularizer=reg)(feat_input)
    out = layers.Dropout(0.3)(out)
    pred = layers.Dense(1, activation='sigmoid', dtype='float32', name='task_output')(out)
    classifier = keras.Model(inputs=feat_input, outputs=pred, name='classifier')
    return feature_extractor, classifier

# ------------------ Main function per seed ------------------
def run_single_model(seed, num_labeled=200):
    np.random.seed(seed)
    tf.random.set_seed(seed)
    import random
    random.seed(seed)

    # Load and prepare data (original, no imputation)
    df = pd.read_csv('/kaggle/input/datasets/urospoleksic/full-dataset-output-csv/full_dataset_output.csv')
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    target_cir_cols = [f'CIR{i}' for i in range(716, 1016)]
    tabular_cols = ['RANGE','FP_IDX','FP_AMP1','FP_AMP2','FP_AMP3','STDEV_NOISE','CIR_PWR','MAX_NOISE','RXPACC','CH','FRAME_LEN','PREAM_LEN','BITRATE','PRFR']
    required_cols = target_cir_cols + tabular_cols + ['NLOS','DOMAIN']
    df.dropna(subset=required_cols, inplace=True)

    train_domains = [1,2]
    test_domain = 3
    df_src = df[df['DOMAIN'].isin(train_domains)].copy()
    df_tgt = df[df['DOMAIN'] == test_domain].copy()

    scaler = StandardScaler()
    df_src[tabular_cols] = scaler.fit_transform(df_src[tabular_cols])
    df_tgt[tabular_cols] = scaler.transform(df_tgt[tabular_cols])

    df_src_val = df_src.sample(frac=0.15, random_state=42)
    df_src_train = df_src.drop(df_src_val.index)

    def process_data(df_input):
        cirs = df_input[target_cir_cols].values.astype(np.float32)
        max_cir_amp = np.max(np.abs(cirs), axis=1, keepdims=True)
        X_cir = np.expand_dims(cirs / (max_cir_amp + 1e-9), axis=-1)
        X_tab = df_input[tabular_cols].values.astype(np.float32)
        y = df_input['NLOS'].values.astype(np.float32)
        return X_cir, X_tab, y

    X_cir_src_tr, X_tab_src_tr, y_src_tr = process_data(df_src_train)
    X_cir_src_val, X_tab_src_val, y_src_val = process_data(df_src_val)
    X_cir_tgt_all, X_tab_tgt_all, y_tgt_all = process_data(df_tgt)

    # Target split
    rng = np.random.RandomState(42)   # fixed split across seeds
    indices = np.arange(len(df_tgt))
    rng.shuffle(indices)
    labeled_idx = indices[:num_labeled]
    test_idx = indices[num_labeled:]
    X_cir_tgt_labeled = X_cir_tgt_all[labeled_idx]
    X_tab_tgt_labeled = X_tab_tgt_all[labeled_idx]
    y_tgt_labeled = y_tgt_all[labeled_idx]
    X_cir_tgt_test = X_cir_tgt_all[test_idx]
    X_tab_tgt_test = X_tab_tgt_all[test_idx]
    y_tgt_test = y_tgt_all[test_idx]

    # Create datasets
    batch_size = 256
    src_train_ds = tf.data.Dataset.from_tensor_slices((X_cir_src_tr, X_tab_src_tr, y_src_tr)).shuffle(20000).batch(batch_size, drop_remainder=True)
    tgt_train_ds = tf.data.Dataset.from_tensor_slices((X_cir_tgt_all, X_tab_tgt_all)).shuffle(20000).repeat().batch(batch_size, drop_remainder=True)
    train_dataset = tf.data.Dataset.zip((src_train_ds, tgt_train_ds)).prefetch(tf.data.AUTOTUNE)
    val_dataset = tf.data.Dataset.from_tensor_slices(((X_cir_src_val, X_tab_src_val), y_src_val)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    # Build and train CORAL
    feature_extractor, classifier = build_backbone(len(tabular_cols))
    coral_model = CORALUDAModel(feature_extractor, classifier, max_coral_weight=500.0, max_entropy_weight=0.0)
    epochs = 50
    lr_schedule = keras.optimizers.schedules.CosineDecay(5e-4, epochs*(len(X_cir_src_tr)//batch_size), alpha=0.01)
    coral_model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr_schedule))
    class CoralScheduler(keras.callbacks.Callback):
        def __init__(self, total_epochs):
            super().__init__()
            self.total_epochs = total_epochs
        def on_epoch_begin(self, epoch, logs=None):
            self.model.set_epoch(epoch, self.total_epochs)
    history = coral_model.fit(
        train_dataset,
        validation_data=val_dataset,
        steps_per_epoch=len(X_cir_src_tr)//batch_size,
        epochs=epochs,
        callbacks=[
            keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True, mode='max'),
            CoralScheduler(epochs)
        ],
        verbose=0
    )
    best_val_acc = max(history.history['val_accuracy']) * 100

    # Fine-tune (original simple)
    cir_input_ft = keras.Input(shape=(300,1), name='cir_input_ft')
    tab_input_ft = keras.Input(shape=(len(tabular_cols),), name='tab_input_ft')
    feats = feature_extractor([cir_input_ft, tab_input_ft])
    preds = classifier(feats)
    finetune_model = keras.Model(inputs=[cir_input_ft, tab_input_ft], outputs=preds)
    finetune_model.compile(optimizer=keras.optimizers.Adam(1e-5), loss='binary_crossentropy', metrics=['accuracy'])

    X_cir_ft = np.concatenate([X_cir_src_tr, X_cir_tgt_labeled], axis=0)
    X_tab_ft = np.concatenate([X_tab_src_tr, X_tab_tgt_labeled], axis=0)
    y_ft = np.concatenate([y_src_tr, y_tgt_labeled], axis=0)
    ft_idx = np.arange(len(X_cir_ft))
    np.random.shuffle(ft_idx)
    X_cir_ft, X_tab_ft, y_ft = X_cir_ft[ft_idx], X_tab_ft[ft_idx], y_ft[ft_idx]
    finetune_model.fit([X_cir_ft, X_tab_ft], y_ft, batch_size=64, epochs=10, verbose=0)

    # TTA prediction
    def predict_with_tta(X_cir, X_tab, n_aug=5):
        preds = []
        for _ in range(n_aug):
            cir_noisy = X_cir + np.random.normal(0, 0.01, size=X_cir.shape).astype(np.float32)
            shift = np.random.randint(-3, 4)
            cir_shifted = np.roll(cir_noisy, shift, axis=1)
            tab_noisy = X_tab + np.random.normal(0, 0.01, size=X_tab.shape).astype(np.float32)
            feats = feature_extractor([cir_shifted, tab_noisy], training=False)
            pred = classifier(feats, training=False).numpy().flatten()
            preds.append(pred)
        return np.mean(preds, axis=0)

    tgt_preds = predict_with_tta(X_cir_tgt_test, X_tab_tgt_test)
    binary_preds = (tgt_preds > 0.5).astype(np.int32)
    seed_target_acc = np.mean(binary_preds == y_tgt_test) * 100

    print(f"Seed {seed}: Source val acc = {best_val_acc:.2f}% | Target test acc (TTA) = {seed_target_acc:.2f}%")

    return tgt_preds, y_tgt_test, best_val_acc

# ------------------ Ensemble Main ------------------
if __name__ == '__main__':
    seeds = [42, 123, 456]
    all_preds = []
    all_val_accs = []
    y_test_all = None

    for seed in seeds:
        print(f"Running seed {seed}...")
        preds, y_test, val_acc = run_single_model(seed, num_labeled=200)
        all_preds.append(preds)
        all_val_accs.append(val_acc)
        if y_test_all is None:
            y_test_all = y_test
        else:
            assert np.array_equal(y_test_all, y_test), "Test labels differ!"

    avg_preds = np.mean(all_preds, axis=0)
    binary = (avg_preds > 0.5).astype(np.int32)
    ensemble_acc = np.mean(binary == y_test_all) * 100

    print("\n--- Final Ensemble Summary ---")
    for i, seed in enumerate(seeds):
        print(f"Seed {seed}: source val acc = {all_val_accs[i]:.2f}%")
    print(f"Ensemble Target Test Accuracy (TTA): {ensemble_acc:.2f}%")

Running seed 42...


I0000 00:00:1788090824.147167      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1788090824.150302      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1788090855.499090     137 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Seed 42: Source val acc = 93.57% | Target test acc (TTA) = 75.23%
Running seed 123...
Seed 123: Source val acc = 93.52% | Target test acc (TTA) = 73.70%
Running seed 456...
Seed 456: Source val acc = 93.65% | Target test acc (TTA) = 73.99%

--- Final Ensemble Summary ---
Seed 42: source val acc = 93.57%
Seed 123: source val acc = 93.52%
Seed 456: source val acc = 93.65%
Ensemble Target Test Accuracy (TTA): 74.34%
